# multi_1h_jma_atr_020

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (183).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `1h` |
| Code cells | 13 |
| Detected functions | resample_1hr_custom, compute_jma_indicator, cy_process_ticker, backtest_trades, calculate_metrics_fast, apply_strategy |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%load_ext cython

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Data/Indices_Data_merged.csv')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2021-01-01'
end_date = '2024-12-31'

# Define the tickers you want to filter
selected_tickers = ['USA30IDXUSD60', 'USA500IDXUSD60', 'USATECHIDXUSD60']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

def compute_jma_indicator(
    df,
    smooth_length=21,
    normalization_period=24,
    upper_threshold=1,
    lower_threshold=-1.5,
    rs_period=3,
    rs_percentile=50
):
    # === Jurik Moving Average (JMA) Calculation ===
    price_series = df['close']
    alpha = 2.0 / (smooth_length + 1)

    jma_values = [price_series.iloc[0]]
    for price in price_series.iloc[1:]:
        jma_values.append(jma_values[-1] + (price - jma_values[-1]) * alpha)

    jma_series = pd.Series(jma_values, index=price_series.index)

    # === Normalization ===
    rolling_mean = jma_series.rolling(window=normalization_period).mean()
    rolling_std = jma_series.rolling(window=normalization_period).std()
    normalized_jma = (jma_series - rolling_mean) / rolling_std

    # === Rogers-Satchell Volatility Calculation ===
    open_prices = df['open'].replace(0, np.nan)
    high_prices = df['high']
    low_prices = df['low']
    close_prices = df['close'].replace(0, np.nan)

    log_hc = np.log(high_prices / close_prices)
    log_ho = np.log(high_prices / open_prices)
    log_lc = np.log(low_prices / close_prices)
    log_lo = np.log(low_prices / open_prices)

    rs_volatility = (log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()

    # === RS Volatility Filter ===
    rs_threshold = rs_volatility.rolling(window=normalization_period).quantile(rs_percentile / 100.0)

    # === Signal Generation ===
    long_signals = (normalized_jma > upper_threshold) & (rs_volatility > rs_threshold)
    short_signals = (normalized_jma < lower_threshold) & (rs_volatility > rs_threshold)

    signals = pd.Series(0, index=df.index)
    signals[long_signals] = 1
    signals[short_signals] = 2

    # === Attach Results to DataFrame ===
    df['JMA'] = jma_series
    df['Normalized_JMA'] = normalized_jma
    df['RS_Volatility'] = rs_volatility
    df['RS_Threshold'] = rs_threshold
    df['Signal'] = signals

    return df

data['Date'] = pd.to_datetime(data['Date']).dt.to_pydatetime()
data = data.groupby('Ticker', group_keys=False).apply(compute_jma_indicator)
data['Signal'] = data.groupby('Ticker')['Signal'].shift(1)
data.dropna(subset=['Signal'], inplace=True)
data['Signal'] = data['Signal'].astype(np.int8)

In [ ]:
data

In [ ]:
%%cython
import numpy as np
cimport numpy as np
from libc.math cimport log

def cy_process_ticker(
    object[:, :] data,
    str ticker,
    double capital,
    double fee_rate,
    double slippage_rate,
    double tp_pct,
    double sl_pct,
    double entry_esl_pct=0.009,
    double universal_sl_pct=0.009
):
    cdef:
        int i, n = data.shape[0]
        bint position_open = False
        double entry_price = 0.0
        object entry_time = None
        int direction = 0
        double quantity = 0.0
        double cumulative_pnl = 0.0
        object last_exit_time = None
        bint sl_tp_check_active = False
        list trades = []

        double tp_price, sl_price, esl_price, exit_price
        double pnl, fee, net_pnl, unrealized_loss_pct
        bint tp_hit, sl_hit, esl_hit
        object time
        double open_, high, low, close
        int signal

    for i in range(n):
        time = data[i, 0]
        open_ = <double>data[i, 1]
        high = <double>data[i, 2]
        low = <double>data[i, 3]
        close = <double>data[i, 4]
        signal = <int>data[i, 5]

        # === ENTRY LOGIC ===
        if not position_open and time != last_exit_time:
            if signal == 1:
                direction = 1
            elif signal == 2:
                direction = -1
            else:
                continue

            entry_price = open_ * (1 + slippage_rate) if direction == 1 else open_ * (1 - slippage_rate)
            quantity = capital / entry_price
            entry_time = time
            position_open = True
            sl_tp_check_active = False

            tp_price = entry_price * (1 + tp_pct) if direction == 1 else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == 1 else entry_price * (1 + sl_pct)
            esl_price = entry_price * (1 - entry_esl_pct) if direction == 1 else entry_price * (1 + entry_esl_pct)

            # === ENTRY ESL CHECK ===
            esl_hit = (low <= esl_price) if direction == 1 else (high >= esl_price)
            if esl_hit:
                exit_price = esl_price * (1 - slippage_rate) if direction == 1 else esl_price * (1 + slippage_rate)
                pnl = (exit_price - entry_price) * quantity if direction == 1 else (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "Entry ESL", cumulative_pnl, "long" if direction == 1 else "short"))
                position_open = False
                last_exit_time = time
                continue

            # === ENTRY TP CHECK ===
            tp_hit = (high >= tp_price) if direction == 1 else (low <= tp_price)
            if tp_hit:
                exit_price = tp_price * (1 - slippage_rate) if direction == 1 else tp_price * (1 + slippage_rate)
                pnl = (exit_price - entry_price) * quantity if direction == 1 else (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "Entry TP", cumulative_pnl, "long" if direction == 1 else "short"))
                position_open = False
                last_exit_time = time
                continue

            sl_tp_check_active = True
            continue

        # === POSITION MANAGEMENT ===
        if position_open:
            tp_price = entry_price * (1 + tp_pct) if direction == 1 else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == 1 else entry_price * (1 + sl_pct)

            tp_hit = (high >= tp_price) if direction == 1 else (low <= tp_price)
            sl_hit = (low <= sl_price) if direction == 1 else (high >= sl_price)

            # === SL CHECK ===
            if sl_tp_check_active and sl_hit:
                exit_price = open_ * (1 - slippage_rate) if direction == 1 else open_ * (1 + slippage_rate)
                pnl = (exit_price - entry_price) * quantity if direction == 1 else (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "Stop Loss", cumulative_pnl, "long" if direction == 1 else "short"))
                position_open = False
                last_exit_time = time
                continue

            # === TP CHECK ===
            if tp_hit:
                exit_price = tp_price * (1 - slippage_rate) if direction == 1 else tp_price * (1 + slippage_rate)
                pnl = (exit_price - entry_price) * quantity if direction == 1 else (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "Take Profit", cumulative_pnl, "long" if direction == 1 else "short"))
                position_open = False
                last_exit_time = time
                continue

            # === UNIVERSAL SL CHECK ===
            unrealized_loss_pct = (close - entry_price) / entry_price if direction == 1 else (entry_price - close) / entry_price
            if unrealized_loss_pct <= -universal_sl_pct:
                exit_price = close * (1 - slippage_rate) if direction == 1 else close * (1 + slippage_rate)
                pnl = (exit_price - entry_price) * quantity if direction == 1 else (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "Universal SL (-0.9%)", cumulative_pnl, "long" if direction == 1 else "short"))
                position_open = False
                last_exit_time = time
                continue

            # === END OF WEEK EXIT ===
            if time.weekday() == 4 and time.hour == 22:
                exit_price = close * (1 - slippage_rate) if direction == 1 else close * (1 + slippage_rate)
                pnl = (exit_price - entry_price) * quantity if direction == 1 else (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "End of Week Exit (Fri 22:00)", cumulative_pnl, "long" if direction == 1 else "short"))
                position_open = False
                last_exit_time = time
                continue

            sl_tp_check_active = True

    return trades


In [ ]:
def backtest_trades(data: pd.DataFrame, tp_pct: float, sl_pct: float) -> pd.DataFrame:
    initial_capital = 100000
    fee = 0.00002
    slippage = 0.00005

    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = np.array(data['Date'].dt.to_pydatetime())  # Fix future warning

    tickers = data['Ticker'].unique()

    ticker_map = {
        ticker: data[data['Ticker'] == ticker][["Date", "open", "high", "low", "close", "Signal"]].to_numpy(dtype=object)
        for ticker in tickers
    }

    # ✅ Use single-threaded loop instead of joblib
    all_trades = []
    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = cy_process_ticker(
            ticker_map[ticker], ticker, initial_capital, fee, slippage, tp_pct, sl_pct
        )
        all_trades.extend(trades)

    return pd.DataFrame(all_trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])


# === Example usage ===
# Run the backtest
tradebook = backtest_trades(data, tp_pct=0.01, sl_pct=0.002)

# Save or display results
tradebook.to_csv("/content/drive/MyDrive/JMA_Tradebook_FX_Indices_T5_2020_25_1.csv", index=False)
tradebook


In [ ]:
from itertools import product
from tqdm import tqdm
import pandas as pd
import numpy as np

# === High-Speed Metrics Calculation ===
def calculate_metrics_fast(tradebook):
    if tradebook.empty:
        return 0.0, 0.0, 0.0, 0.0, 0.0, 0

    dates = pd.to_datetime(tradebook['Entry Time'].values, errors='coerce')
    pnls = tradebook['PnL'].values

    # Cumulative PnL (Equity Curve)
    equity = np.cumsum(pnls)
    total_profit = equity[-1]

    # Drawdown & Ratio
    roll_max = np.maximum.accumulate(equity)
    drawdown = roll_max - equity
    max_drawdown = np.max(drawdown)
    ratio = total_profit / max_drawdown if max_drawdown > 0 else (float('inf') if total_profit > 0 else 0.0)

    # R² - using formula without sklearn
    x = np.arange(len(equity))
    y = equity
    x_mean = x.mean()
    y_mean = y.mean()
    ss_tot = np.sum((y - y_mean) ** 2)
    ss_res = np.sum((y - ((np.dot(x, y) / np.dot(x, x)) * x)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0.0

    # Monthly Consistency
    months = dates.to_numpy().astype('datetime64[M]')
    monthly_pnls = {}
    for m, p in zip(months, pnls):
        monthly_pnls[m] = monthly_pnls.get(m, 0.0) + p
    monthly_vals = np.fromiter(monthly_pnls.values(), dtype=float)
    if len(monthly_vals) < 2:
        monthly_consistency = 0.0
    else:
        m_mean = monthly_vals.mean()
        m_std = monthly_vals.std()
        monthly_consistency = m_mean / m_std if m_std > 0 else float('inf')

    # Win Rate and Total Trades
    total_trades = len(pnls)
    win_rate = np.sum(pnls > 0) / total_trades if total_trades > 0 else 0.0

    return ratio, total_profit, r2, monthly_consistency, win_rate, total_trades

# === Hyperparameter Grid ===
param_grid = {
    'smooth_length': [14, 21, 28, 32],
    'norm_period': [12, 18, 24, 32],
    'upper_threshold': [1, 1.5, 2, 2.5],
    'lower_threshold': [-1.5, -2, -2.5],
    'rs_period': [3, 5],
    'rs_percentile': [30, 50, 70],
    'tp_pct': [0.005, 0.007, 0.009, 0.01],
    'sl_pct': [0.001, 0.0015, 0.002]
}

# === Generate All Parameter Combinations ===
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))
results = []

print(f"Total combinations to test: {len(param_combinations)}\n")

# === Hyperparameter Tuning Loop ===
for i, combo in enumerate(tqdm(param_combinations, desc="Hypertuning"), 1):
    param = dict(zip(keys, combo))

    # Avoid deepcopy unless required
    temp_data = data

    def apply_strategy(group):
        return compute_jma_indicator(
            group,
            smooth_length=param['smooth_length'],
            normalization_period=param['norm_period'],
            upper_threshold=param['upper_threshold'],
            lower_threshold=param['lower_threshold'],
            rs_period=param['rs_period'],
            rs_percentile=param['rs_percentile']
        )

    temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

    # Shift signals to next bar to avoid lookahead bias
    temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
    temp_data = temp_data.dropna(subset=['Signal'])
    temp_data['Signal'] = temp_data['Signal'].astype(int)

    # Run Backtest
    tradebook = backtest_trades(
        temp_data,
        sl_pct=param['sl_pct'],
        tp_pct=param['tp_pct']
    )

    # Evaluate metrics
    ratio, total_profit, r2, monthly_consistency, win_rate, total_trades = calculate_metrics_fast(tradebook)

    # Optional: reduce print frequency
    print(f"\n=== [{i}/{len(param_combinations)}] Params ===")
    for k, v in param.items():
        print(f"{k}: {v}")
    print(f"➡️ Profit/Drawdown: {ratio:.4f} | Profit: ₹{total_profit:,.2f} | R²: {r2:.4f} | Monthly Score: {monthly_consistency:.4f} | Win Rate: {win_rate:.2%} | Trades: {total_trades}")

    result = {
        **param,
        'Profit/Drawdown': ratio,
        'Total Profit': total_profit,
        'Equity R2': r2,
        'Monthly Consistency': monthly_consistency,
        'Win Rate': win_rate,
        'Total Trades': total_trades
    }
    results.append(result)

# === Final Results and Export ===
results_df = pd.DataFrame(results)

# Optional: Composite metric for sorting
results_df['Composite Score'] = (
    results_df['Profit/Drawdown'] * 0.2 +
    results_df['Equity R2'] * 0.5 +
    results_df['Monthly Consistency'] * 0.3
)

results_df = results_df.sort_values(by='Composite Score', ascending=False)
results_df.to_csv("/content/drive/MyDrive/hyperparameter_results_adaptive_intraday_full.csv", index=False)

print("\n✅ Hyperparameter tuning complete. Top 5 results:")
print(results_df.head())
